# Milestone 1 — Google Colab Benchmark & Pipeline Runner (CPU-Only)

**Mục tiêu của Notebook này:**
1. Kiểm tra thông số phần cứng của máy ảo **Google Colab (chế độ CPU-only)**: CPU, RAM khả dụng, dung lượng ổ đĩa.
2. Chạy toàn bộ dây chuyền thu thập dữ liệu (Discovery $\rightarrow$ Download $\rightarrow$ Extract $\rightarrow$ Metrics) trên tập bài báo thử nghiệm (**Pilot**) hoặc toàn bộ khoảng 5.000 ID của từng thành viên.
3. Tự động đo Thời gian chạy, Average / Peak RAM, Peak / Final Disk bằng `ResourceMonitor`.
4. Xuất file kết quả `data/metrics/benchmark_summary.json` và `data/metrics/per_paper_metrics.json` để đưa vào **Jupyter Book** tạo bảng và biểu đồ tự động.

## Bước 1: Chuẩn bị môi trường

In [3]:
import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/theanhnek/arxiv-data-project.git"

if IS_COLAB:
    print("[Colab] Đang chạy trên Google Colab...")
    repo_dir = Path("/content/arxiv-data-project")
    if not repo_dir.exists():
        !git clone {REPO_URL} {repo_dir}
    os.chdir(repo_dir)
    !pip install -q -r src/requirements.txt
else:
    # Nếu mở trực tiếp trong VS Code trên máy tính cá nhân
    if Path.cwd().name == "src":
        os.chdir(Path.cwd().parent)
    print(f"[Local] Đang chạy trên máy cá nhân tại thư mục: {Path.cwd()}")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

[Local] Đang chạy trên máy cá nhân tại thư mục: d:\arxiv-data-project


## Bước 2: (Tùy chọn) Kết nối Google Drive để không bị mất dữ liệu khi tắt máy ảo

Nếu chạy trên Google Colab với số lượng bài lớn có thể đổi `USE_GOOGLE_DRIVE = True` để lưu thẳng thư mục `data/` vào Google Drive của bạn. Khi mạng ngắt hoặc hết phiên Colab, chỉ cần chạy lại với `RESUME = True` là chương trình sẽ làm tiếp từ chỗ đang dở.

In [9]:
USE_GOOGLE_DRIVE = False 

if IS_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/arxiv-data-project/data")
else:
    OUTPUT_DIR = Path("data")

for sub in ("corpus", "state", "metrics", "tmp"):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"Thư mục lưu dữ liệu: {OUTPUT_DIR.resolve()}")

Thư mục lưu dữ liệu: D:\arxiv-data-project\data


## Bước 3: Kiểm tra cấu hình phần cứng (CPU-only Testbed Audit)

In [10]:
import json
from src.metrics import collect_environment_info

env_info = collect_environment_info(OUTPUT_DIR)
print("=== THÔNG SỐ MÔI TRƯỜNG CHẠY BENCHMARK ===")
print(json.dumps(env_info, indent=2, ensure_ascii=False))

=== THÔNG SỐ MÔI TRƯỜNG CHẠY BENCHMARK ===
{
  "python_version": "3.13.5",
  "platform": "Windows-11-10.0.26200-SP0",
  "machine": "AMD64",
  "cpu_count_logical": 16,
  "cpu_count_physical": 8,
  "system_total_ram_bytes": 33492422656,
  "system_total_ram_gb": 31.19,
  "filesystem_total_bytes": 699696934912,
  "filesystem_free_bytes": 352490135552,
  "git_commit": "bc31d0f",
  "is_colab": false
}


## Bước 4: Chọn chế độ chạy (Chạy thử Pilot 5 bài hoặc Chạy theo phân công thành viên)

- **`RUN_MODE = "PILOT"`**: Chạy thử nghiệm nhanh trên **5 bài báo đầu tiên** trong khoảng ID của Thành viên 5 (`2509.11047` $\rightarrow$ `2509.11051`) — chỉ mất khoảng 30–45 giây để kiểm tra toàn bộ pipeline và sinh file `benchmark_summary.json`.
- **`RUN_MODE = "FULL_MEMBER"`**: Chạy toàn bộ 5.000 ID của thành viên được chọn qua `MEMBER_INDEX` (`1..5`).

In [ ]:
from src.discovery import expand_assigned_range

# Bảng phân công 5 thành viên
ASSIGNED_RANGES = {
    1: ("2025-10", 14401, "2025-10", 19400),    #Kiệt
    2: ("2025-10", 9401,  "2025-10", 14400),    #Dương
    3: ("2025-10", 4401,  "2025-10", 9400),     #Hà
    4: ("2025-09", 26047, "2025-10", 4400),     #Kiên   
    5: ("2025-09", 11047, "2025-09", 16046),    #Anh
}

RUN_MODE = "PILOT"      # Đổi thành "FULL_MEMBER" khi chạy thật toàn bộ 5.000 ID
MEMBER_INDEX = 5        # Số thứ tự thành viên (1..5)
PILOT_COUNT = 5         # Số bài báo chạy thử khi ở chế độ PILOT
RESUME = True           # True: tiếp tục từ kết quả đã chạy trước đó
CLEANUP_TMP_AFTER_EXTRACT = False  # Đổi thành True nếu muốn xóa file nén .tar.gz sau khi giải nén để tiết kiệm ổ cứng Colab

start_m, start_id, end_m, end_id = ASSIGNED_RANGES[MEMBER_INDEX]
all_member_ids = expand_assigned_range(start_m, start_id, end_m, end_id, expected_total=5000)

if RUN_MODE == "PILOT":
    target_ids = all_member_ids[:PILOT_COUNT]
    print(f"[Chế độ PILOT] Chọn {len(target_ids)} bài đầu tiên của Thành viên {MEMBER_INDEX}: {target_ids[0]} -> {target_ids[-1]}")
else:
    target_ids = all_member_ids
    print(f"[Chế độ FULL_MEMBER] Chọn toàn bộ {len(target_ids)} bài của Thành viên {MEMBER_INDEX}: {target_ids[0]} -> {target_ids[-1]}")

[Chế độ PILOT] Chọn 5 bài đầu tiên của Thành viên 5: 2509.11047 -> 2509.11051


## Bước 5: Chạy Pipeline End-to-End và Đo Hiệu năng (Runtime, RAM, Disk)

In [7]:
import os
import shutil
from src.config import CrawlConfig
from src.discovery import ArxivClient, discover_papers, write_json
from src.downloader import download_discovered_sources
from src.extractor import extract_downloaded_sources
from src.references import collect_references
from src.storage import export_tables
from src.metrics import ResourceMonitor, compute_pipeline_metrics

config = CrawlConfig(
    output_dir=OUTPUT_DIR,
    arxiv_delay_seconds=3.0,
    request_timeout_seconds=60.0,
    max_attempts=3,
    user_agent=os.getenv("ARXIV_USER_AGENT", "Lab1ArxivCrawler/0.1 (HCMUS Intro-DS Group)"),
    metrics_sample_interval_seconds=0.5,
)

write_json(
    OUTPUT_DIR / "state" / "input_validation.json",
    {"ids": target_ids, "invalid": [], "duplicates": []},
)

client = ArxivClient(config)
with ResourceMonitor(config.output_dir, config.metrics_sample_interval_seconds) as monitor:
    try:
        print("--- [1/5] Bắt đầu Discovery ---")
        disc_report = discover_papers(target_ids, config, resume=RESUME, client=client)

        print("\n--- [2/5] Bắt đầu Download ---")
        dl_report = download_discovered_sources(config, resume=RESUME, client=client)

        print("\n--- [3/5] Bắt đầu Extraction (.tex / .bib) ---")
        ext_report = extract_downloaded_sources(config, resume=RESUME)

        print("\n--- [4/5] Bắt đầu References (Semantic Scholar) ---")
        ref_report = collect_references(config, resume=RESUME)

        print("\n--- [5/5] Xuất 2 bảng dữ liệu (papers.jsonl & references.jsonl) ---")
        export_report = export_tables(config)
    finally:
        client.close()

# Nếu bật dọn dẹp file nén tạm sau khi đã đo Peak Disk và giải nén xong
if CLEANUP_TMP_AFTER_EXTRACT:
    tmp_dl_dir = OUTPUT_DIR / "tmp" / "downloads"
    if tmp_dl_dir.exists():
        shutil.rmtree(tmp_dl_dir, ignore_errors=True)
        print("Đã dọn dẹp thư mục file nén tạm data/tmp/downloads/ sau khi giải nén.")

summary_metrics = compute_pipeline_metrics(config, monitor_snapshot=monitor.snapshot)
print("\n=== HOÀN TẤT ĐO LƯỜNG! ĐÃ LƯU BÁO CÁO TẠI data/metrics/benchmark_summary.json ===")
print(f"Xuất bảng thành công: {export_report['papers_rows']} bài và {export_report['references_rows']} trích dẫn.")

--- [1/3] Bắt đầu Discovery ---
[Discovery 1/5] 2509.11047: cached (1 versions)
[Discovery 2/5] 2509.11048: resolved (1 versions)
[Discovery 3/5] 2509.11049: resolved (2 versions)
[Discovery 4/5] 2509.11050: resolved (1 versions)
[Discovery 5/5] 2509.11051: resolved (1 versions)

--- [2/3] Bắt đầu Download ---
[Download 1/6] 2509.11047v1: downloaded (588904 bytes)
[Download 2/6] 2509.11048v1: failed (0 bytes)
[Download 3/6] 2509.11049v1: downloaded (16246 bytes)
[Download 4/6] 2509.11049v2: downloaded (16927 bytes)
[Download 5/6] 2509.11050v1: downloaded (4527094 bytes)
[Download 6/6] 2509.11051v1: failed (0 bytes)

--- [3/3] Bắt đầu Extraction (.tex / .bib) ---
[Extract 1/6] 2509.11047v1: cached (7 files)
[Extract 3/6] 2509.11049v1: success (2 files, 48619 bytes)
[Extract 4/6] 2509.11049v2: success (2 files, 50748 bytes)
[Extract 5/6] 2509.11050v1: success (1 files, 37833 bytes)

=== HOÀN TẤT ĐO LƯỜNG! ĐÃ LƯU BÁO CÁO TẠI data/metrics/benchmark_summary.json ===


## Bước 6: Xem Bảng tổng kết Kết quả & Hiệu năng

In [8]:
disc = summary_metrics["discovery"]
dlext = summary_metrics["download_and_extraction"]
sizes = summary_metrics["sizes_bytes"]
perf = summary_metrics["performance"]
res = perf["resources"]
refs = summary_metrics.get("references") or {}

print(f"1. Số bài báo kiểm tra (Assigned IDs)        : {disc['assigned_unique_ids']}")
print(f"2. Số bài báo tìm thấy (Resolved Papers)     : {disc['resolved_papers']} ({disc['paper_discovery_success_rate']*100:.2f}%)")
print(f"3. Tổng số phiên bản (Discovered Versions)   : {disc['discovered_versions']}")
print(f"4. Số phiên bản tải & giải nén thành công    : {dlext['extracted_tex_versions']} có TeX + {dlext['no_tex_source_versions']} không có TeX ({dlext['version_completion_rate']*100:.2f}%)")
print(f"5. Kích thước TB 1 bài TRƯỚC khi bỏ ảnh      : {sizes['avg_uncompressed_bytes_per_paper_before_filtering']:,.0f} bytes")
print(f"6. Kích thước TB 1 bài SAU khi bỏ ảnh (.tex) : {sizes['avg_retained_bytes_per_paper_after_filtering']:,.0f} bytes")
print(f"7. Tỷ lệ tiết kiệm dung lượng nhờ bỏ ảnh     : {sizes['figure_removal_savings_ratio']*100:.2f}%")
print("-" * 65)
if refs and refs.get("available"):
    print(f"8. Số bài query References Semantic Scholar  : {refs['queried_papers']} ({refs['reference_scrape_success_rate']*100:.2f}%)")
    print(f"9. Tổng số trích dẫn thu thập được           : {refs['total_references_crawled']} (TB {refs['avg_references_per_queried_paper']:.1f} refs/bài)")
    print(f"10. Số trích dẫn chứa arXiv ID hợp lệ        : {refs['total_arxiv_references_crawled']}")
    print("-" * 65)
print(f"11. Thời gian chạy Discovery                 : {perf['discovery_elapsed_seconds']:.2f} giây")
print(f"12. Tổng thời gian chạy toàn tuyến (Wall time): {perf['end_to_end_wall_time_seconds']:.2f} giây")
print(f"13. Thời gian xử lý trung bình 1 bài báo     : {perf['avg_time_per_resolved_paper_seconds']:.2f} giây/bài")
print(f"14. Bộ nhớ RAM trung bình / cực đại          : {res['avg_ram_mb']:.2f} MB / {res['peak_ram_mb']:.2f} MB")
print(f"15. Dung lượng ổ đĩa đỉnh (Peak Disk)        : {res['peak_working_disk_mb']:.3f} MB")
print(f"16. Dung lượng Corpus cuối cùng (Final Disk) : {res['final_corpus_disk_mb']:.3f} MB")

1. Số bài báo kiểm tra (Assigned IDs)        : 5
2. Số bài báo tìm thấy (Resolved Papers)     : 5 (100.00%)
3. Tổng số phiên bản (Discovered Versions)   : 6
4. Số phiên bản tải & giải nén thành công    : 4 có TeX + 0 không có TeX (66.67%)
5. Kích thước TB 1 bài TRƯỚC khi bỏ ảnh      : 1,653,873 bytes
6. Kích thước TB 1 bài SAU khi bỏ ảnh (.tex) : 34,012 bytes
7. Tỷ lệ tiết kiệm dung lượng nhờ bỏ ảnh     : 97.94%
-----------------------------------------------------------------
8. Thời gian chạy Discovery                  : 10.70 giây
9. Tổng thời gian chạy toàn tuyến (Wall time): 128.82 giây
10. Thời gian xử lý trung bình 1 bài báo     : 25.77 giây/bài
11. Bộ nhớ RAM trung bình / cực đại          : 97.25 MB / 98.78 MB
12. Dung lượng ổ đĩa đỉnh (Peak Disk)        : 7.062 MB
13. Dung lượng Corpus cuối cùng (Final Disk) : 0.164 MB
